#Implement Q-learning
Q-learning is a value-based reinforcement learning algorithm that learns the optimal Q-values for state-action pairs through exploration and exploitation. With a solid understanding of Q-learning and its theoretical basis, let's move on to the practical implementation of the algorithm, starting with the initialization of the Q-table.

#Step-by-step guide:
#Step 1: Initialize the Q-table
In this step, we initialized the Q-table to store the values for each possible state-action pair. The table had 25 rows (one for each state in a 5x5 grid) and 4 columns (one for each action: up, down, left, and right).

In [ ]:
import numpy as np

grid_size = 5
n_actions = 4

# Initialize Q-table with zeros
Q_table = np.zeros((grid_size * grid_size, n_actions))

#Step 2: Define the reward matrix
The agent received a reward of +10 for reaching the goal (state 24), a penalty of –10 for falling into the pit (state 12), and –1 for every other action to encourage faster exploration of the grid.#

In [ ]:
rewards = np.full((grid_size * grid_size,), -1)
rewards[24] = 10  # Goal state
rewards[12] = -10  # Pitfall state

#Step 3: Implement the epsilon-greedy policy
To balance exploration and exploitation, we used an epsilon-greedy strategy. With probability ϵ, the agent chose a random action, and with probability 1−ϵ, it chose the best-known action based on the Q-values.

In [ ]:
def epsilon_greedy_action(Q_table, state, epsilon):
    if np.random.uniform(0, 1) < epsilon:
        return np.random.randint(0, n_actions)  # Explore
    else:
        return np.argmax(Q_table[state])  # Exploit

#Step 4: Q-learning update rule
The Q-values were updated based on the Bellman equation:

Q
(
s
,
a
)
←
Q
(
s
,
a
)
+
α
[
r
+
γ
max
⁡
a
′
Q
(
s
′
,
a
′
)
−
Q
(
s
,
a
)
]
Q(s,a)←Q(s,a)+α[r+γ
a
′

max
​
 Q(s
′
 ,a
′
 )−Q(s,a)]
Q, left parenthesis, s, comma, a, right parenthesis, left arrow, Q, left parenthesis, s, comma, a, right parenthesis, plus, alpha, open bracket, r, plus, gamma, \max, start subscript, a, prime, end subscript, Q, left parenthesis, s, prime, comma, a, prime, right parenthesis, minus, Q, left parenthesis, s, comma, a, right parenthesis, close bracket
Where:

Q(s, a) is the current Q-value for the state-action pair,

s is the current state,

a is the action taken in the current state,

r is the immediate reward received for the current action,

γ is the discount factor for future rewards,

α is the learning rate,

s′ is the new state, and

a′ is the action that maximizes the future reward in the new state.

In [ ]:
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor
epsilon = 0.1  # Exploration rate

for episode in range(1000):
    state = np.random.randint(0, grid_size * grid_size)  # Random start
    done = False
    while not done:
        action = epsilon_greedy_action(Q_table, state, epsilon)
        next_state = np.random.randint(0, grid_size * grid_size)  # Random next state
        reward = rewards[next_state]

        # Update Q-value using Bellman equation
        Q_table[state, action] = Q_table[state, action] + alpha * (reward + gamma * np.max(Q_table[next_state]) - Q_table[state, action])

        state = next_state
        if next_state == 24 or next_state == 12:
            done = True  # End episode if goal or pitfall is reached

#Policy gradient implementation
Policy gradients are a policy-based reinforcement learning method where the agent directly learns a policy by maximizing the probability of actions that lead to higher rewards.

#Step-by-step guide:
Step 1: Build the policy network
In this step, we built a neural network using TensorFlow to model the policy. The network took the current state as input and output action probabilities using a softmax activation function.

In [ ]:
import tensorflow as tf

n_states = grid_size * grid_size  # 25 states in the grid
n_actions = 4  # Four possible actions

model = tf.keras.Sequential([
    tf.keras.layers.Dense(24, activation='relu', input_shape=(n_states,)),
    tf.keras.layers.Dense(n_actions, activation='softmax')  # Output action probabilities
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#Step 2: Action selection
For each state, the agent selected an action based on the probabilities outputted by the policy network.

In [ ]:
def get_action(state):
    state_input = tf.one_hot(state, n_states)  # One-hot encode the state
    action_probs = model(state_input[np.newaxis, :])
    return np.random.choice(n_actions, p=action_probs.numpy()[0])

#Step 3: Cumulative rewards calculation
To give more weight to actions that led to long-term success, we computed cumulative rewards for each time step during an episode.

In [ ]:
def compute_cumulative_rewards(rewards, gamma=0.99):
    cumulative_rewards = np.zeros_like(rewards)
    running_add = 0
    for t in reversed(range(len(rewards))):
        running_add = running_add * gamma + rewards[t]
        cumulative_rewards[t] = running_add
    return cumulative_rewards

#Step 4: Update policy using REINFORCE
We used the REINFORCE algorithm to update the policy. The loss function was the negative log-probability of the actions taken, scaled by the cumulative rewards.

In [ ]:
def update_policy(states, actions, rewards):
    cumulative_rewards = compute_cumulative_rewards(rewards)

    with tf.GradientTape() as tape:
        state_inputs = tf.one_hot(states, n_states)
        action_probs = model(state_inputs)
        action_masks = tf.one_hot(actions, n_actions)

        # Log-probabilities of the actions taken
        log_probs = tf.reduce_sum(action_masks * tf.math.log(action_probs), axis=1)

        # Policy loss function
        loss = -tf.reduce_mean(log_probs * cumulative_rewards)

    # Apply gradients to update the policy network
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))